In [4]:
import os
from pathlib import Path
from abc import ABC
from dataclasses import dataclass
from typing import List
from fogvis.db import Database
from contextlib import closing

DB_PATH : Path = Path(os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir, "media", "db")))
db = Database(DB_PATH)

In [2]:
@dataclass
class FogImage:
    image_path: str
    vis_distance: str


@dataclass
class Dataset:
    images: List[FogImage]
    
class Experiment(ABC):
    def __init__(self, db: Database) -> None:
        self.db: Database = db

    def get_datasets(self) -> List[Dataset]:
        """Gathers a list of datasets to be used for training.
        Each item refers to a different set of data to be used in training a
        seperate instance of a model

        Returns:
            list: _description_
        """
        pass

class Linear_vs_Exponential(Experiment):
    def __init__(self, db: Database) -> None:
        super().__init__(db)

    @staticmethod
    def Get_Fog_Images(
        db: Database, fog_type_name: str, scene_name: str
    ) -> list[FogImage]:
        result: list[FogImage] = []
        with closing(db.get_connection().cursor()) as ctx:
            sql: str = """SELECT image.filePath, image.visibilityDistance from image 
            INNER JOIN scene ON image.sceneID = scene.id 
            INNER JOIN camera ON image.cameraID = camera.id
            INNER JOIN environment ON image.environmentID = environment.id
            INNER JOIN fog ON environment.fogID = fog.id
            INNER JOIN fog_type ON fog.typeID = fog_type.id
            WHERE 
                fog_type.name = ? AND
                scene.name = ?
            """
            parms = (fog_type_name, scene_name)
            ctx.execute(sql, parms)
            res = ctx.fetchall()
            if len(res) == 0:
                raise Exception("Failed to get images for scene")

            for r in res:
                result.append(FogImage(image_path=r[0], vis_distance=r[1]))

        return result

    @staticmethod
    def Get_Linear_Dataset(db: Database) -> Dataset:
        return Dataset(
            images=Linear_vs_Exponential.Get_Fog_Images(
                db, "linear", "berthoud_pass_co"
            )
        )

    @staticmethod
    def Get_Exp_Fog_Images(db: Database) -> Dataset:
        return Dataset(
            images=Linear_vs_Exponential.Get_Fog_Images(
                db, "exponential", "berthoud_pass_co"
            )
        )

    def get_datasets(self) -> List[Dataset]:
        result: List[Dataset] = [
            self.Get_Linear_Dataset(self.db),
            self.Get_Exp_Fog_Images(self.db),
        ]
        return result


In [8]:
experiment : Linear_vs_Exponential = Linear_vs_Exponential(db)
datasets = experiment.get_datasets()